In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.pipeline import Pipeline

In [ ]:
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report
)

def evaluate_model(pipe, X_train, y_train, X_test, y_test, plot_train_roc=True, metrics_print=True):
    """
    Evaluate a trained classification pipeline (or model) on train and test data.

    Prints metrics for both classes (0 & 1), and macro/weighted F1.

    Returns:
    --------
    metrics_df : pd.DataFrame
        Precision, Recall, F1, Weighted F1, Macro F1, and ROC-AUC for Train and Test sets.
    """

    # === Predictions ===
    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    if plot_train_roc:
        # === Confusion Matrices ===
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)
    
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        sns.heatmap(cm_train, annot=True, fmt='d', cmap='Greens', ax=axes[0])
        axes[0].set_title('Train Confusion Matrix')
        axes[0].set_xlabel('Predicted')
        axes[0].set_ylabel('Actual')
    
        sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', ax=axes[1])
        axes[1].set_title('Test Confusion Matrix')
        axes[1].set_xlabel('Predicted')
        axes[1].set_ylabel('Actual')
    
        plt.tight_layout()
        plt.show()

    # === Predict probabilities for ROC-AUC ===
    y_train_proba = pipe.predict_proba(X_train)[:, 1]
    y_test_proba = pipe.predict_proba(X_test)[:, 1]

    # === Compute metrics ===
    def get_detailed_metrics(y_true, y_pred, y_proba):
        report = classification_report(y_true, y_pred, output_dict=True)
        precision_0 = report['0']['precision']
        recall_0 = report['0']['recall']
        f1_0 = report['0']['f1-score']

        precision_1 = report['1']['precision']
        recall_1 = report['1']['recall']
        f1_1 = report['1']['f1-score']

        macro_f1 = f1_score(y_true, y_pred, average='macro')
        weighted_f1 = f1_score(y_true, y_pred, average='weighted')
        auc = roc_auc_score(y_true, y_proba)

        return {
            'Precision_0': precision_0, 'Recall_0': recall_0, 'F1_0': f1_0,
            'Precision_1': precision_1, 'Recall_1': recall_1, 'F1_1': f1_1,
            'F1_macro': macro_f1, 'F1_weighted': weighted_f1, 'ROC AUC': auc
        }

    train_metrics = get_detailed_metrics(y_train, y_train_pred, y_train_proba)
    test_metrics = get_detailed_metrics(y_test, y_test_pred, y_test_proba)

    # === Metrics Table ===
    metrics_df = pd.DataFrame([
        {'Dataset': 'Train', **train_metrics},
        {'Dataset': 'Test', **test_metrics}
    ]).round(4)

    if metrics_print:
        display(metrics_df)

    # === Optional: ROC Curve for Train ===
    if plot_train_roc:
        fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr_train, tpr_train, color='darkorange', lw=2,
                 label=f'Train ROC (AUC = {train_metrics["ROC AUC"]:.3f})')
        plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('Train ROC Curve')
        plt.legend(loc='lower right')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.show()

    return metrics_df


In [ ]:
image_folder = "images_transformed"   # keep if you need it elsewhere
csv_path = "grid_features.csv"        # your merged features CSV
save_results_path = "feature_experiments_results.csv"
merged_df = pd.read_csv(csv_path)

In [ ]:
from sklearn.model_selection import train_test_split

# ==== Unique image filenames ====
unique_images = merged_df['filename'].unique()

# ==== Split by images ====
train_imgs, test_imgs = train_test_split(
    unique_images,
    test_size=0.15,
    random_state=42,
    shuffle=True  # set False if you don’t want randomization
)

train_df = merged_df[merged_df["filename"].isin(train_imgs)].reset_index(drop=True)
test_df  = merged_df[merged_df["filename"].isin(test_imgs)].reset_index(drop=True)

print(f"\nTotal images: {len(unique_images)}")
print(f"Train images: {len(train_imgs)} | Test images: {len(test_imgs)}")
print(f"Train grids: {len(train_df)} | Test grids: {len(test_df)}")

# --- Check label distribution ---
label_counts = train_df["label"].value_counts()
label_percent = 100 * label_counts / len(train_df)
print("\nLabel distribution in training data:")
print(pd.DataFrame({"count": label_counts, "percent": label_percent.round(2)}))

# --- Final feature/label split ---
X_train = train_df.drop(columns=["filename", "label"])
y_train = train_df["label"]

X_test = test_df.drop(columns=["filename", "label"])
y_test = test_df["label"]

print("\n✅ Train/test feature sets ready:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")


In [ ]:

use_cols = [
    'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',

    # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y'
]

# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # 60% of majority → balanced but not overdone
        k_neighbors=3,          # smaller neighborhood for modest datasets
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth= 20,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight='balanced',      # SMOTE handles imbalance
        random_state=42,
        n_jobs=-1
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
features_used = X_train_reduced.columns.tolist()
metrics_with_features = metrics.copy()
metrics_with_features['features_used'] = [features_used] * len(metrics_with_features)

In [ ]:

use_cols = [
    'row', 'col',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',

    # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y'
]

# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # 60% of majority → balanced but not overdone
        k_neighbors=3,          # smaller neighborhood for modest datasets
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth= 20,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight='balanced',      # SMOTE handles imbalance
        random_state=42,
        n_jobs=-1
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
entry = metrics.copy()
entry['features_used'] = [features_used] * len(entry)
metrics_with_features = pd.concat([metrics_with_features, entry], ignore_index=True)

In [ ]:

use_cols = [
    'row', 'col', 'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',

    # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y'
]

# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # 60% of majority → balanced but not overdone
        k_neighbors=3,          # smaller neighborhood for modest datasets
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth= 20,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight='balanced',      # SMOTE handles imbalance
        random_state=42,
        n_jobs=-1
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
entry = metrics.copy()
entry['features_used'] = [features_used] * len(entry)
metrics_with_features = pd.concat([metrics_with_features, entry], ignore_index=True)

In [ ]:
use_cols = [
    'row', 'col', 'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',

    # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y', 'edge_density'
]
# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

features_used = (X_train_reduced.columns.tolist())

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # 60% of majority → balanced but not overdone
        k_neighbors=3,          # smaller neighborhood for modest datasets
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth= 20,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight='balanced',      # SMOTE handles imbalance
        random_state=42,
        n_jobs=-1
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
entry = metrics.copy()
entry['features_used'] = [features_used] * len(entry)
metrics_with_features = pd.concat([metrics_with_features, entry], ignore_index=True)


In [ ]:
use_cols = [
    'row', 'col', 'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',

    # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y', 'hs_ratio', 'hv_contrast'
]
# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

features_used = (X_train_reduced.columns.tolist())

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # 60% of majority → balanced but not overdone
        k_neighbors=3,          # smaller neighborhood for modest datasets
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth= 20,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight='balanced',      # SMOTE handles imbalance
        random_state=42,
        n_jobs=-1
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
entry = metrics.copy()
entry['features_used'] = [features_used] * len(entry)
metrics_with_features = pd.concat([metrics_with_features, entry], ignore_index=True)


In [ ]:
corr = X_train_reduced.corr().abs()
plt.figure(figsize=(10,8))
sns.heatmap(corr, vmin=0, vmax=1)
plt.title("Feature correlation (abs)")
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# === Sobel feature columns ===
neighbor_cols = [
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y'
]

def add_neighbor_features(df, n_cols=8):
    df = df.copy()
    new_features = []

    for fname in df["filename"].unique():
        img_df = df[df["filename"] == fname].copy().reset_index(drop=True)

        # Build lookup for (row, col) → feature values
        lookup = img_df.set_index(["row", "col"])

        # Compute neighbor mean/std for each block
        for idx, row in img_df.iterrows():
            r, c = row["row"], row["col"]

            # 8-connected neighborhood (excluding self)
            neighbors = [
                (r + dr, c + dc)
                for dr in [-1, 0, 1]
                for dc in [-1, 0, 1]
                if not (dr == 0 and dc == 0)
                and 0 <= r + dr < n_cols
                and 0 <= c + dc < n_cols
            ]

            # Collect neighbor feature rows
            nbr_vals = []
            for (nr, nc) in neighbors:
                if (nr, nc) in lookup.index:
                    nbr_vals.append(lookup.loc[(nr, nc), neighbor_cols])

            # Compute mean and std from neighbor values
            if nbr_vals:
                nbr_df = pd.DataFrame(nbr_vals)
                for col in neighbor_cols:
                    img_df.loc[idx, f"{col}_nbrmean"] = nbr_df[col].mean()
                    img_df.loc[idx, f"{col}_nbrstd"] = nbr_df[col].std()
            else:
                for col in neighbor_cols:
                    img_df.loc[idx, f"{col}_nbrmean"] = np.nan
                    img_df.loc[idx, f"{col}_nbrstd"] = np.nan

        new_features.append(img_df)

    # Combine all processed image frames
    df_final = pd.concat(new_features, ignore_index=True)
    return df_final


# === Apply to merged_df and save ===
merged_df = add_neighbor_features(merged_df, n_cols=8)
merged_df.to_csv(csv_path, index=False)

print("✅ Added Sobel neighbor mean/std features using existing row/col and saved to:", csv_path)


In [ ]:
use_cols = [
    'row', 'col', 'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',

    # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y', 
    'sobel_mean_mag_nbrmean', 'sobel_mean_mag_nbrstd',
    'sobel_std_mag_nbrmean',  'sobel_std_mag_nbrstd',
    'sobel_mean_x_nbrmean',   'sobel_mean_x_nbrstd',
    'sobel_std_x_nbrmean',    'sobel_std_x_nbrstd',
    'sobel_mean_y_nbrmean',   'sobel_mean_y_nbrstd',
    'sobel_std_y_nbrmean',    'sobel_std_y_nbrstd'
]
# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

features_used = (X_train_reduced.columns.tolist())

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # 60% of majority → balanced but not overdone
        k_neighbors=3,          # smaller neighborhood for modest datasets
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth= 20,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight='balanced',      # SMOTE handles imbalance
        random_state=42,
        n_jobs=-1
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
entry = metrics.copy()
entry['features_used'] = [features_used] * len(entry)
metrics_with_features = pd.concat([metrics_with_features, entry], ignore_index=True)


In [ ]:

use_cols = [
    'row', 'col', 'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',

    # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y'
]

# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

print("✅ Columns after excluding GLCM and including neighbor stats:")
print(X_train_reduced.columns.tolist())

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # 60% of majority → balanced but not overdone
        k_neighbors=3,          # smaller neighborhood for modest datasets
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth= 20,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight='balanced',      # SMOTE handles imbalance
        random_state=42,
        n_jobs=-1
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
entry = metrics.copy()
entry['features_used'] = [features_used] * len(entry)
metrics_with_features = pd.concat([metrics_with_features, entry], ignore_index=True)


In [ ]:
# === Select only saliency + HSV + their neighbor mean/std features ===
use_cols = [
    'row', 'col', 'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',

    # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
]

# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

print("✅ Columns after excluding GLCM and including neighbor stats:")
print(X_train_reduced.columns.tolist())

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # 60% of majority → balanced but not overdone
        k_neighbors=3,          # smaller neighborhood for modest datasets
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth= 20,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight='balanced',      # SMOTE handles imbalance
        random_state=42,
        n_jobs=-1
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
entry = metrics.copy()
entry['features_used'] = [features_used] * len(entry)
metrics_with_features = pd.concat([metrics_with_features, entry], ignore_index=True)


In [ ]:
# === Select only saliency + HSV + their neighbor mean/std features ===
use_cols = [
    'row', 'col', 'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',

    # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y'
]

# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

print("✅ Columns after excluding GLCM and including neighbor stats:")
print(X_train_reduced.columns.tolist())

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # 60% of majority → balanced but not overdone
        k_neighbors=3,          # smaller neighborhood for modest datasets
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth= 20,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight='balanced',      # SMOTE handles imbalance
        random_state=42,
        n_jobs=-1
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    True,  # disable plot
    True    # enable classification report
)


In [ ]:
corr = X_train_reduced.corr().abs()
plt.figure(figsize=(10,8))
sns.heatmap(corr, vmin=0, vmax=1)
plt.title("Feature correlation (abs)")
plt.show()

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# === Config ===
image_folder = "images_transformed"
output_folder = "rf_test_results_2"
os.makedirs(output_folder, exist_ok=True)

# === Add predictions to test_df ===
test_df['predicted'] = pipe.predict(X_test_reduced)

# === Grid configuration ===
grid_rows, grid_cols = 8, 8
img_w, img_h = 800, 600
block_w = img_w // grid_cols
block_h = img_h // grid_rows

# === Visualization parameters ===
overlay_color = (0, 255, 255)   # yellow in BGR
alpha = 0.5                     # transparency (0 → fully transparent, 1 → opaque)

# === Process each unique image ===
for fname in tqdm(test_df['filename'].unique(), desc="Saving visualized results"):
    img_path = os.path.join(image_folder, fname)
    if not os.path.exists(img_path):
        print(f"⚠️ Missing: {img_path}")
        continue

    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Failed to load: {fname}")
        continue

    # overlay copy
    overlay = img.copy()

    # subset of current image blocks
    img_blocks = test_df[test_df['filename'] == fname]

    for _, row in img_blocks.iterrows():
        block_id = int(row['block_id'])
        pred = int(row['predicted'])

        # Compute grid row, col
        r = block_id // grid_cols
        c = block_id % grid_cols

        # Block boundaries
        x1, y1 = c * block_w, r * block_h
        x2, y2 = x1 + block_w, y1 + block_h

        # Apply yellow overlay if predicted = 1
        if pred == 1:
            cv2.rectangle(overlay, (x1, y1), (x2, y2), overlay_color, -1)

        # Draw grid boundary lines (thin gray)
        cv2.rectangle(img, (x1, y1), (x2, y2), (150, 150, 150), 1)

    # Blend overlay (alpha for yellow, original for rest)
    vis_img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

    # Save
    save_path = os.path.join(output_folder, fname)
    cv2.imwrite(save_path, vis_img)

print("✅ Visualization complete! Saved to:", output_folder)


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# === Config ===
image_folder = "images_transformed"
output_folder = "rf_train_results_2"
os.makedirs(output_folder, exist_ok=True)

# === Add predictions to test_df ===
train_df['predicted'] = pipe.predict(X_train_reduced)

# === Grid configuration ===
grid_rows, grid_cols = 8, 8
img_w, img_h = 800, 600
block_w = img_w // grid_cols
block_h = img_h // grid_rows

# === Visualization parameters ===
overlay_color = (0, 255, 255)   # yellow in BGR
alpha = 0.5                     # transparency (0 → fully transparent, 1 → opaque)

# === Process each unique image ===
for fname in tqdm(train_df['filename'].unique(), desc="Saving visualized results"):
    img_path = os.path.join(image_folder, fname)
    if not os.path.exists(img_path):
        print(f"⚠️ Missing: {img_path}")
        continue

    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Failed to load: {fname}")
        continue

    # overlay copy
    overlay = img.copy()

    # subset of current image blocks
    img_blocks = train_df[train_df['filename'] == fname]

    for _, row in img_blocks.iterrows():
        block_id = int(row['block_id'])
        pred = int(row['predicted'])

        # Compute grid row, col
        r = block_id // grid_cols
        c = block_id % grid_cols

        # Block boundaries
        x1, y1 = c * block_w, r * block_h
        x2, y2 = x1 + block_w, y1 + block_h

        # Apply yellow overlay if predicted = 1
        if pred == 1:
            cv2.rectangle(overlay, (x1, y1), (x2, y2), overlay_color, -1)

        # Draw grid boundary lines (thin gray)
        cv2.rectangle(img, (x1, y1), (x2, y2), (150, 150, 150), 1)

    # Blend overlay (alpha for yellow, original for rest)
    vis_img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

    # Save
    save_path = os.path.join(output_folder, fname)
    cv2.imwrite(save_path, vis_img)

print("✅ Visualization complete! Saved to:", output_folder)


In [ ]:
display(metrics_with_features)
metrics_with_features.to_csv('metrics_with_features.csv', index =False)

In [ ]:
from xgboost import XGBClassifier
use_cols = [
    'row', 'col', 'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',

    # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y'
]

# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

features_used = (X_train_reduced.columns.tolist())

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # handle imbalance
        k_neighbors=3,
        random_state=42
    )),
    ('xgb', XGBClassifier(
        n_estimators=300,        # same as RF
        max_depth=20,            # same tree depth as RF
        learning_rate=0.05,      # gentle step size for better generalization
        subsample=0.8,           # like RF’s random sampling
        colsample_bytree=0.8,    # feature subsampling like RF’s randomness
        min_child_weight=3,      # similar to min_samples_split/leaf
        gamma=0.0,               # min loss reduction for splits
        reg_lambda=1.0,          # L2 regularization
        scale_pos_weight=1.0,    # balanced due to SMOTE
        n_jobs=-1,
        random_state=42,
        objective='binary:logistic',  # assuming binary classification
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
entry = metrics.copy()
entry["features_used"] = [features_used] * len(entry)

# Create new DataFrame instead of appending to metrics_with_features
xgb_metrics = pd.concat([pd.DataFrame(), entry], ignore_index=True)


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from tqdm import tqdm
import numpy as np

# === Select only saliency + HSV + their neighbor mean/std features ===
use_cols = [
    'row', 'col', 'block_id',
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y'
]

# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()


# === Define base XGBoost model ===
xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    n_jobs=-1,
    random_state=42
)

# === Define pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,
        k_neighbors=3,
        random_state=42
    )),
    ('xgb', xgb_base)
])

# === Define parameter grid for RandomizedSearch ===
param_dist = {
    'xgb__n_estimators': [200, 300, 400],
    'xgb__max_depth': [4, 6, 8, 10],
    'xgb__learning_rate': [0.05, 0.1, 0.2],
    'xgb__subsample': [0.7, 0.8, 1.0],
    'xgb__colsample_bytree': [0.7, 0.8, 1.0],
    'xgb__min_child_weight': [1, 3, 5],
    'xgb__gamma': [0, 0.1, 0.3],
    'xgb__reg_lambda': [0.5, 1.0, 2.0]
}

# === Use RandomizedSearchCV for faster tuning ===
tqdm.write("🔍 Starting randomized hyperparameter tuning...")

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=15,                # only 15 random combinations for speed
    cv=3,                     # 3-fold cross-validation
    verbose=2,                # show progress
    n_jobs=-1,
    scoring='f1',             # or 'accuracy' depending on your use case
    random_state=42
)

# === Run the search ===
search.fit(X_train_reduced, y_train)

print("\n✅ Best parameters found:")
print(search.best_params_)

# === Train final model on full training data using best params ===
final_pipe = search.best_estimator_
final_pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    final_pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)


In [ ]:
from sklearn.decomposition import PCA
use_cols = [
    'row', 'col', 'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',
     # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y', 
    'sobel_mean_mag_nbrmean', 'sobel_mean_mag_nbrstd',
    'sobel_std_mag_nbrmean',  'sobel_std_mag_nbrstd',
    'sobel_mean_x_nbrmean',   'sobel_mean_x_nbrstd',
    'sobel_std_x_nbrmean',    'sobel_std_x_nbrstd',
    'sobel_mean_y_nbrmean',   'sobel_mean_y_nbrstd',
    'sobel_std_y_nbrmean',    'sobel_std_y_nbrstd', 
       'hs_ratio'	,'hv_contrast'

   
]
# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

features_used = (X_train_reduced.columns.tolist())

# === Define the pipeline ===

pipe = Pipeline([
    ('scaler', StandardScaler()),

    

    ('smote', SMOTE(
        sampling_strategy=0.5,  # handle imbalance
        k_neighbors=3,
        random_state=42
    )),

    ('xgb', XGBClassifier(
        n_estimators=300,        # same as RF
        max_depth=20,            # same tree depth as RF
        learning_rate=0.05,      # gentle step size for better generalization
        subsample=0.8,           # like RF’s random sampling
        colsample_bytree=0.8,    # feature subsampling like RF’s randomness
        min_child_weight=3,      # similar to min_samples_split/leaf
        gamma=0.0,               # min loss reduction for splits
        reg_lambda=1.0,          # L2 regularization
        scale_pos_weight=1.0,    # balanced due to SMOTE
        n_jobs=-1,
        random_state=42,
        objective='binary:logistic',  # assuming binary classification
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
# Initialize only once before loop
xgb_metrics = pd.DataFrame()

# Inside your loop:
entry = metrics.copy()
entry["features_used"] = [features_used] * len(entry)
xgb_metrics = pd.concat([xgb_metrics, entry], ignore_index=True)


In [ ]:
use_cols = [
    'row', 'col', 'block_id',

    # base features
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',
     # neighbor features
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y', 
    'sobel_mean_mag_nbrmean', 'sobel_mean_mag_nbrstd',
    'sobel_std_mag_nbrmean',  'sobel_std_mag_nbrstd',
    'sobel_mean_x_nbrmean',   'sobel_mean_x_nbrstd',
    'sobel_std_x_nbrmean',    'sobel_std_x_nbrstd',
    'sobel_mean_y_nbrmean',   'sobel_mean_y_nbrstd',
    'sobel_std_y_nbrmean',    'sobel_std_y_nbrstd', 

]
# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

features_used = (X_train_reduced.columns.tolist())

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,  # handle imbalance
        k_neighbors=3,
        random_state=42
    )),
    ('xgb', XGBClassifier(
        n_estimators=300,        # same as RF
        max_depth=20,            # same tree depth as RF
        learning_rate=0.05,      # gentle step size for better generalization
        subsample=0.8,           # like RF’s random sampling
        colsample_bytree=0.8,    # feature subsampling like RF’s randomness
        min_child_weight=3,      # similar to min_samples_split/leaf
        gamma=0.0,               # min loss reduction for splits
        reg_lambda=1.0,          # L2 regularization
        scale_pos_weight=1.0,    # balanced due to SMOTE
        n_jobs=-1,
        random_state=42,
        objective='binary:logistic',  # assuming binary classification
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
# Initialize only once before loop
xgb_metrics = pd.DataFrame()

# Inside your loop:
entry = metrics.copy()
entry["features_used"] = [features_used] * len(entry)
xgb_metrics = pd.concat([xgb_metrics, entry], ignore_index=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# === Define the feature columns ===
use_cols = [
    'row', 'col', 'block_id',
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y',
    'sobel_mean_mag_nbrmean', 'sobel_mean_mag_nbrstd',
    'sobel_std_mag_nbrmean',  'sobel_std_mag_nbrstd',
    'sobel_mean_x_nbrmean',   'sobel_mean_x_nbrstd',
    'sobel_std_x_nbrmean',    'sobel_std_x_nbrstd',
    'sobel_mean_y_nbrmean',   'sobel_mean_y_nbrstd',
    'sobel_std_y_nbrmean',    'sobel_std_y_nbrstd',
    'hs_ratio', 'hv_contrast', 'lap_mean', 'lap_std'
]

# === Prepare feature data ===
X = merged_df[use_cols].copy()

# Remove identifiers before VIF
id_cols = ['row', 'col', 'block_id']
X_num = X.drop(columns=id_cols)

# Standardize for stability
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_num), columns=X_num.columns)

# === Function to calculate VIF ===
def calculate_vif(df):
    vif_data = pd.DataFrame()
    vif_data["feature"] = df.columns
    vif_data["VIF"] = [
        variance_inflation_factor(df.values, i)
        for i in range(df.shape[1])
    ]
    return vif_data.sort_values(by="VIF", ascending=False)

# === Iteratively remove features with VIF > 10 ===
vif_threshold = 10
X_iter = X_scaled.copy()

while True:
    vif_df = calculate_vif(X_iter)
    high_vif = vif_df[vif_df["VIF"] > vif_threshold]
    if high_vif.empty:
        print("✅ All remaining features have VIF ≤ 10.")
        break
    to_remove = high_vif.iloc[0]["feature"]
    print(f"⚠️ Removing '{to_remove}' (VIF={high_vif.iloc[0]['VIF']:.2f})")
    X_iter = X_iter.drop(columns=[to_remove])

# === Final retained features ===
final_features = X_iter.columns.tolist()
print("\n✅ Final features after VIF filtering (≤ 10):")
print(final_features)

# === Plot the final VIF values ===
final_vif = calculate_vif(X_iter)
plt.figure(figsize=(10, 6))
plt.barh(final_vif['feature'], final_vif['VIF'], color='skyblue')
plt.axvline(10, color='r', linestyle='--', label='VIF = 10 Threshold')
plt.xlabel('VIF Value')
plt.ylabel('Feature')
plt.title('Final VIF Values (after filtering)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# === Define columns to use ===
use_cols = [
    'row', 'col', 'block_id',
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y',
    'sobel_mean_mag_nbrmean', 'sobel_mean_mag_nbrstd',
    'sobel_std_mag_nbrmean',  'sobel_std_mag_nbrstd',
    'sobel_mean_x_nbrmean',   'sobel_mean_x_nbrstd',
    'sobel_std_x_nbrmean',    'sobel_std_x_nbrstd',
    'sobel_mean_y_nbrmean',   'sobel_mean_y_nbrstd',
    'sobel_std_y_nbrmean',    'sobel_std_y_nbrstd',
    'hs_ratio', 'hv_contrast', 'lap_mean', 'lap_std'
]

# === Prepare data ===
X = merged_df[use_cols].select_dtypes(include=[np.number])
y = merged_df['label'].astype(int)

# === Standardize features ===
X_scaled = StandardScaler().fit_transform(X)

# === Run t-SNE ===
print("🚀 Running t-SNE... this might take a moment.")

# Instantiate TSNE with minimal params (compatible everywhere)
tsne = TSNE(
    n_components=2,
    random_state=42,
    perplexity=30
)

# Some old sklearn versions accept `n_iter` only inside fit_transform
try:
    X_embedded = tsne.fit_transform(X_scaled)
except TypeError:
    # Fallback for extremely old sklearn versions
    X_embedded = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_scaled)

# === Plot ===
plt.figure(figsize=(8, 6))
plt.scatter(
    X_embedded[y == 0, 0], X_embedded[y == 0, 1],
    c='royalblue', label='Label 0', alpha=0.6, s=20
)
plt.scatter(
    X_embedded[y == 1, 0], X_embedded[y == 1, 1],
    c='tomato', label='Label 1', alpha=0.6, s=20
)
plt.title("t-SNE Visualization of Selected Features")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.decomposition import PCA
use_cols = [
    'block_id','row', 'col','sal_avg', 'sal_std', 'h_mean', 's_mean', 'h_std', 's_std', 'v_std', 'sal_avg_nbrmean', 'sal_avg_nbrstd', 'sal_std_nbrmean', 
    'sal_std_nbrstd', 'h_mean_nbrmean', 'h_mean_nbrstd', 's_mean_nbrmean', 's_mean_nbrstd', 'v_mean_nbrmean', 'v_mean_nbrstd', 'h_std_nbrmean', 
    'h_std_nbrstd', 's_std_nbrmean', 's_std_nbrstd', 'v_std_nbrmean', 'v_std_nbrstd', 'sobel_mean_x', 'sobel_std_x', 'sobel_mean_y', 'sobel_std_y',
    'sobel_mean_mag_nbrstd', 'sobel_std_mag_nbrstd', 'sobel_mean_x_nbrmean', 'sobel_mean_x_nbrstd', 'sobel_std_x_nbrstd', 'sobel_mean_y_nbrmean',
    'sobel_mean_y_nbrstd', 'sobel_std_y_nbrmean', 'hs_ratio', 'hv_contrast', 'lap_mean', 'lap_std'
   
]
# === Prepare training and testing sets ===
X_train_reduced = X_train[use_cols].copy()
X_test_reduced  = X_test[use_cols].copy()

features_used = (X_train_reduced.columns.tolist())

# === Define the pipeline ===

pipe = Pipeline([
    ('scaler', StandardScaler()),

    

    ('smote', SMOTE(
        sampling_strategy=0.5,  # handle imbalance
        k_neighbors=3,
        random_state=42
    )),

    ('xgb', XGBClassifier(
        n_estimators=300,        # same as RF
        max_depth=20,            # same tree depth as RF
        learning_rate=0.05,      # gentle step size for better generalization
        subsample=0.8,           # like RF’s random sampling
        colsample_bytree=0.8,    # feature subsampling like RF’s randomness
        min_child_weight=3,      # similar to min_samples_split/leaf
        gamma=0.0,               # min loss reduction for splits
        reg_lambda=1.0,          # L2 regularization
        scale_pos_weight=1.0,    # balanced due to SMOTE
        n_jobs=-1,
        random_state=42,
        objective='binary:logistic',  # assuming binary classification
        eval_metric='logloss',
        use_label_encoder=False
    ))
])

# === Train the model ===
pipe.fit(X_train_reduced, y_train)

# === Evaluate the model ===
metrics = evaluate_model(
    pipe,
    X_train_reduced, y_train,
    X_test_reduced, y_test,
    False,  # disable plot
    True    # enable classification report
)
# Initialize only once before loop
xgb_metrics = pd.DataFrame()

# Inside your loop:
entry = metrics.copy()
entry["features_used"] = [features_used] * len(entry)
xgb_metrics = pd.concat([xgb_metrics, entry], ignore_index=True)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Access the trained XGBoost model from the pipeline
xgb_model = pipe.named_steps['xgb']

# Get the feature names from your reduced dataset
feature_names = X_train_reduced.columns

# === Compute importances ===
importances = xgb_model.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# ✅ Print all features in descending order
print("🎯 Feature Importances (All Features, Sorted Descending):")
for i, (feature, importance) in enumerate(zip(feat_imp['Feature'], feat_imp['Importance']), 1):
    print(f"{i:2d}. {feature:<40} {importance:.6f}")

# === Visualize (unchanged) ===
plt.figure(figsize=(10, 6))
plt.barh(feat_imp['Feature'][:20], feat_imp['Importance'][:20])
plt.gca().invert_yaxis()
plt.title("Top 20 Feature Importances (XGBoost)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import uniform, randint
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# === Define the pipeline ===
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(
        sampling_strategy=0.5,
        k_neighbors=3,
        random_state=42
    )),
    ('xgb', XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        use_label_encoder=False,
        n_jobs=-1,
        random_state=42
    ))
])

# === Revised parameter distributions ===
param_distributions = {
    # Slightly higher range of estimators (more boosting rounds)
    'xgb__n_estimators': randint(300, 800),

    # Allow moderate depth, not too shallow
    'xgb__max_depth': randint(5, 10),

    # Balance between learning stability and model flexibility
    'xgb__learning_rate': uniform(0.03, 0.08),

    # Ensure trees have enough samples to avoid overfitting
    'xgb__min_child_weight': randint(3, 10),

    # Subsampling adds stochasticity — keeps generalization strong
    'xgb__subsample': uniform(0.7, 0.25),         # [0.7, 0.95]
    'xgb__colsample_bytree': uniform(0.7, 0.25),  # [0.7, 0.95]

    # Add moderate to strong regularization for stability
    'xgb__gamma': uniform(0.05, 0.4),             # penalize overly easy splits
    'xgb__reg_lambda': uniform(1.0, 6.0),         # stronger L2
    'xgb__reg_alpha': uniform(0.5, 2.5)           # some L1 sparsity
}

# === Stratified 5-fold CV ===
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# === Randomized Search ===
print("🚀 Starting balanced XGBoost hyperparameter tuning...")
search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    n_iter=40,               # a few more iterations improves exploration
    scoring='f1_macro',      # balanced metric for both classes
    n_jobs=-1,
    cv=cv,
    verbose=2,
    random_state=42
)

# === Run tuning ===
search.fit(X_train_reduced, y_train)

# === Best model summary ===
print("\n✅ Best Hyperparameters:")
for param, value in search.best_params_.items():
    print(f"{param}: {value}")

print(f"\n🏆 Best CV Score (F1-macro): {search.best_score_:.4f}")

# === Final model on full train set ===
best_model = search.best_estimator_
print("\n🧠 Retraining best model on full training set...")
best_model.fit(X_train_reduced, y_train)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_curve,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score
)

def evaluate_model(model, X, y, dataset_name="Dataset"):
    print(f"\n=== Evaluation on {dataset_name} ===")
    
    # Predictions & probabilities
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    
    # 1️⃣ Confusion Matrix
    cm = confusion_matrix(y, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')
    plt.title(f'Confusion Matrix ({dataset_name})')
    plt.show()
    
    # 2️⃣ Classification Report & Accuracy
    print("Classification Report:\n", classification_report(y, y_pred))
    print("Accuracy:", accuracy_score(y, y_pred))
    
    # 3️⃣ Precision–Recall Curve
    precision, recall, _ = precision_recall_curve(y, y_proba)
    avg_precision = average_precision_score(y, y_proba)
    
    plt.figure(figsize=(6,5))
    plt.plot(recall, precision, label=f'PR curve (AP={avg_precision:.3f})', color='blue')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision–Recall Curve ({dataset_name})')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # 4️⃣ ROC Curve
    fpr, tpr, _ = roc_curve(y, y_proba)
    roc_auc = roc_auc_score(y, y_proba)
    
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f'ROC curve (AUC={roc_auc:.3f})', color='darkorange')
    plt.plot([0,1], [0,1], 'k--', label='Random Guess')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve ({dataset_name})')
    plt.legend()
    plt.grid(True)
    plt.show()


# === Evaluate on Train Set ===
evaluate_model(best_model, X_train_reduced, y_train, dataset_name="Train Set")

# === Evaluate on Test Set ===
evaluate_model(best_model, X_test_reduced, y_test, dataset_name="Test Set")


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# === Config ===
image_folder = "images_transformed"
output_folder = "xgb_test_results"
os.makedirs(output_folder, exist_ok=True)

# === Add predictions to test_df ===
test_df['predicted'] = best_model.predict(X_test_reduced)

# === Grid configuration ===
grid_rows, grid_cols = 8, 8
img_w, img_h = 800, 600
block_w = img_w // grid_cols
block_h = img_h // grid_rows

# === Visualization parameters ===
overlay_color = (0, 255, 255)   # yellow in BGR
alpha = 0.5                     # transparency (0 → fully transparent, 1 → opaque)

# === Process each unique image ===
for fname in tqdm(test_df['filename'].unique(), desc="Saving visualized results"):
    img_path = os.path.join(image_folder, fname)
    if not os.path.exists(img_path):
        print(f"⚠️ Missing: {img_path}")
        continue

    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Failed to load: {fname}")
        continue

    # overlay copy
    overlay = img.copy()

    # subset of current image blocks
    img_blocks = test_df[test_df['filename'] == fname]

    for _, row in img_blocks.iterrows():
        block_id = int(row['block_id'])
        pred = int(row['predicted'])

        # Compute grid row, col
        r = block_id // grid_cols
        c = block_id % grid_cols

        # Block boundaries
        x1, y1 = c * block_w, r * block_h
        x2, y2 = x1 + block_w, y1 + block_h

        # Apply yellow overlay if predicted = 1
        if pred == 1:
            cv2.rectangle(overlay, (x1, y1), (x2, y2), overlay_color, -1)

        # Draw grid boundary lines (thin gray)
        cv2.rectangle(img, (x1, y1), (x2, y2), (150, 150, 150), 1)

    # Blend overlay (alpha for yellow, original for rest)
    vis_img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

    # Save
    save_path = os.path.join(output_folder, fname)
    cv2.imwrite(save_path, vis_img)

print("✅ Visualization complete! Saved to:", output_folder)


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# === Config ===
image_folder = "images_transformed"
output_folder = "XGB_train_results"
os.makedirs(output_folder, exist_ok=True)

# === Add predictions to test_df ===
train_df['predicted'] = best_model.predict(X_train_reduced)

# === Grid configuration ===
grid_rows, grid_cols = 8, 8
img_w, img_h = 800, 600
block_w = img_w // grid_cols
block_h = img_h // grid_rows

# === Visualization parameters ===
overlay_color = (0, 255, 255)   # yellow in BGR
alpha = 0.5                     # transparency (0 → fully transparent, 1 → opaque)

# === Process each unique image ===
for fname in tqdm(train_df['filename'].unique(), desc="Saving visualized results"):
    img_path = os.path.join(image_folder, fname)
    if not os.path.exists(img_path):
        print(f"⚠️ Missing: {img_path}")
        continue

    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Failed to load: {fname}")
        continue

    # overlay copy
    overlay = img.copy()

    # subset of current image blocks
    img_blocks = train_df[train_df['filename'] == fname]

    for _, row in img_blocks.iterrows():
        block_id = int(row['block_id'])
        pred = int(row['predicted'])

        # Compute grid row, col
        r = block_id // grid_cols
        c = block_id % grid_cols

        # Block boundaries
        x1, y1 = c * block_w, r * block_h
        x2, y2 = x1 + block_w, y1 + block_h

        # Apply yellow overlay if predicted = 1
        if pred == 1:
            cv2.rectangle(overlay, (x1, y1), (x2, y2), overlay_color, -1)

        # Draw grid boundary lines (thin gray)
        cv2.rectangle(img, (x1, y1), (x2, y2), (150, 150, 150), 1)

    # Blend overlay (alpha for yellow, original for rest)
    vis_img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

    # Save
    save_path = os.path.join(output_folder, fname)
    cv2.imwrite(save_path, vis_img)

print("✅ Visualization complete! Saved to:", output_folder)


In [ ]:
import pickle

with open("best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)
print("✅ best_model.pkl saved")
